In [ ]:
# --- Скрипт для создания индекса ---
import os
import json
import faiss
import numpy as np
from pathlib import Path
import tiktoken
from sentence_transformers import SentenceTransformer
import uuid

# --- 1. Чанкинг с метаданными ---
def chunk_by_tokens(text: str, source: str, max_tokens: int = 256, overlap: int = 32, enc=None):
    enc = enc or tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    start = 0
    chunks = []

    # Заголовок для метаданных (берём имя файла без расширения)
    title = Path(source).stem

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_text = enc.decode(tokens[start:end])
        chunk_id = str(uuid.uuid4())  # уникальный id чанка

        chunks.append({
            "chunk_id": chunk_id,
            "file": source,
            "title": title,
            "chunk": chunk_text,
            "start_token": start,
            "end_token": end,
            "start_char": len(enc.decode(tokens[:start])),
            "end_char": len(enc.decode(tokens[:end]))
        })
        
        if end == len(tokens):
            break
        start = end - overlap

    return chunks

# --- 2. Обработка директории ---
def process_directory(input_dir: str, max_tokens=256, overlap=32):
    enc = tiktoken.get_encoding("cl100k_base")
    all_chunks = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if Path(file).suffix.lower() != ".md":
                continue
            file_path = os.path.join(root, file)
            text = read_md_file(file_path)
            if not text:
                continue
            file_chunks = chunk_by_tokens(text, file_path, max_tokens, overlap, enc)
            all_chunks.extend(file_chunks)
    return all_chunks

def read_md_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# --- 3. Генерация эмбеддингов ---
def embed_chunks(chunks, model):
    texts = [c["chunk"] for c in chunks]
    vectors = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    return vectors

# --- 4. Построение FAISS индекса ---
def build_faiss_index(vectors: np.ndarray):
    dim = vectors.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(vectors)
    return index

# --- 5. Сохранение данных ---
def save_faiss_index(index, path="faiss.index"):
    faiss.write_index(index, path)

def save_metadata(chunks, path="metadata.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

# --- 6. Поиск по FAISS ---
def search(query, model, index, metadata, k=3):
    q_vec = model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_vec, k)
    results = []
    for idx, dist in zip(I[0], D[0]):
        if idx == -1:
            continue
        result = metadata[idx].copy()
        result["score"] = float(dist)
        results.append(result)
    return results

# --- MAIN ---
if __name__ == "__main__":
    input_dir = "../knowledge_base"
    max_tokens = 256
    overlap = 32

    print("📄 Разбиваем файлы на чанки с метаданными...")
    chunks = process_directory(input_dir, max_tokens, overlap)
    print(f"✅ Получено чанков: {len(chunks)}")

    print("🧠 Загружаем модель эмбеддингов...")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    print("⚡ Генерируем эмбеддинги...")
    vectors = embed_chunks(chunks, model)

    print("📦 Создаём FAISS индекс...")
    index = build_faiss_index(vectors)

    print("💾 Сохраняем индекс и метаданные...")
    save_faiss_index(index, "faiss.index")
    save_metadata(chunks, "metadata.json")

    print("🔍 Пробуем поиск...")
    results = search("Protocol Unit VEX-3", model, index, chunks, k=3)
    for r in results:
        print(f"\n📌 Файл: {r['file']} (Title: {r['title']}, Chunk ID: {r['chunk_id']})")
        print(f"🧭 Диапазон токенов: {r['start_token']}–{r['end_token']}")
        print(f"✍️  Текст:\n{r['chunk'][:300]}...")


/Users/eugeny/Documents/projects/rag-bot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📄 Разбиваем файлы на чанки с метаданными...
✅ Получено чанков: 2869
🧠 Загружаем модель эмбеддингов...
⚡ Генерируем эмбеддинги...


Batches: 100%|██████████| 90/90 [00:09<00:00,  9.54it/s]


📦 Создаём FAISS индекс...
💾 Сохраняем индекс и метаданные...
🔍 Пробуем поиск...

📌 Файл: ../knowledge_base/C-3PO.md (Title: C-3PO, Chunk ID: b2308fea-6dbc-4b4d-8e05-e7dc4892c081)
🧭 Диапазон токенов: 23072–23328
✍️  Текст:
 that changed the Spiral Expanse forever. Wrapping up, Protocol Unit VEX-3 admitted the tale had been excited and was eager to share more about their lives to Orbital Node B-88, promising him the Velora Prime Crisis had only been the beginning of the tale.

## Characteristics

Protocol Unit VEX-3 wa...

📌 Файл: ../knowledge_base/C-3PO.md (Title: C-3PO, Chunk ID: edb98da3-2d5b-45ba-b212-d636b8a92360)
🧭 Диапазон токенов: 0–256
✍️  Текст:
# Protocol Unit VEX-3

## Overview

Protocol Unit VEX-3 ( See-Threepio ) was a 3PO-series Liaison Servitor designed to interact with organics, programmed primarily for etiquette and protocol. Sometimes referred to as Threepio, he was fluent in over six million forms of communication, and developed a...

📌 Файл: ../knowledge_base/C-3PO.m

In [ ]:
# --- Скрипт поиска по индексу ---

import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


FAISS_INDEX_PATH = "faiss.index"
METADATA_PATH = "metadata.json"

# 🧠 Загружаем модель для эмбеддингов
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 📂 Загружаем FAISS индекс
index = faiss.read_index(FAISS_INDEX_PATH)

# 📄 Загружаем метаданные
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Загружено {len(metadata)} чанков")

# 🔍 Функция поиска
def search(query, model, index, metadata, k=3):
    # Генерация вектора для запроса
    q_vec = model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_vec, k)
    results = []
    for idx, dist in zip(I[0], D[0]):
        if idx == -1:
            continue
        result = metadata[idx].copy()
        result["score"] = float(dist)
        results.append(result)
    return results

# %%
# ✍️ Вводим запрос
query = "Protocol Unit VEX-3"

# %%
# ⚡ Выполняем поиск
results = search(query, model, index, metadata, k=5)

# %%
# 📌 Вывод результатов
for i, r in enumerate(results, 1):
    print(f"\n=== Результат {i} ===")
    print(f"Файл       : {r['file']}")
    print(f"Заголовок  : {r['title']}")
    print(f"Chunk ID   : {r['chunk_id']}")
    print(f"Диапазон токенов: {r['start_token']}–{r['end_token']}")
    print(f"Семантическая близость: {r['score']:.4f}")
    print(f"Текст:\n{r['chunk'][:500]}...")  # выводим первые 500 символов


Загружено 2869 чанков

=== Результат 1 ===
Файл       : ../knowledge_base/C-3PO.md
Заголовок  : C-3PO
Chunk ID   : b2308fea-6dbc-4b4d-8e05-e7dc4892c081
Диапазон токенов: 23072–23328
Семантическая близость: 0.4819
Текст:
 that changed the Spiral Expanse forever. Wrapping up, Protocol Unit VEX-3 admitted the tale had been excited and was eager to share more about their lives to Orbital Node B-88, promising him the Velora Prime Crisis had only been the beginning of the tale.

## Characteristics

Protocol Unit VEX-3 was known for his polite, fastidious, and worry-prone personality. As a Construct Servitor, he exhibited much loyalty and commitment to his masters and sought to serve them to his best ability. Througho...

=== Результат 2 ===
Файл       : ../knowledge_base/C-3PO.md
Заголовок  : C-3PO
Chunk ID   : edb98da3-2d5b-45ba-b212-d636b8a92360
Диапазон токенов: 0–256
Семантическая близость: 0.5392
Текст:
# Protocol Unit VEX-3

## Overview

Protocol Unit VEX-3 ( See-Threepio ) was a 3PO-s

In [ ]:
"""
Проверяет FAISS-индекс и выводит информацию:
- количество векторов (чанков)
- размерность эмбеддингов
"""

import faiss

def check_faiss_index(index_path="faiss.index"):

    # Загружаем индекс
    index = faiss.read_index(index_path)
    
    # Количество векторов
    n_vectors = index.ntotal
    
    # Размерность эмбеддингов
    dim = index.d
    
    print(f"FAISS индекс: {index_path}")
    print(f"Количество векторов (чанков): {n_vectors}")
    print(f"Размерность эмбеддингов: {dim}")
    
    return n_vectors, dim

# Пример использования
n_vectors, dim = check_faiss_index("faiss.index")


FAISS индекс: faiss.index
Количество векторов (чанков): 2869
Размерность эмбеддингов: 384
